In [ ]:
import gradio as gr
import torch, requests
from torch import nn
import torch.nn.functional as F
from pathlib import Path
from huggingface_hub import hf_hub_download
import numpy as np
from torch.hub import load_state_dict_from_url

# Serve/Share using a temporary link

## First app

### Create the interface

In [ ]:
def predict(input: str) -> str:
    return "prediction"

In [ ]:
title = "Ask Rick a Question"
description = """
The bot was trained to answer questions based on Rick and Morty dialogues. Ask Rick anything!
<img src="https://huggingface.co/spaces/course-demos/Rick_and_Morty_QA/resolve/main/rick.png" width=200px>
"""

article = "Check out [the original Rick and Morty Bot](https://huggingface.co/spaces/kingabzpro/Rick_and_Morty_Bot) that this demo is based off of."

interface = gr.Interface(
    fn=predict,
    inputs="textbox",
    outputs="text",
    title=title,
    description=description,
    article=article,
    examples=[["What are you doing?"], ["Where should we time travel to?"]],
)

### Create the link

In [ ]:
# interface.launch(share=True)
# # Don't do this. doesn't seem safe !!!

## Second app

### Load artifacts

In [ ]:
repo_id="course-demos/Sketch-Recognition"
hf_space_dir = "https://huggingface.co/spaces"
model_url = f"{hf_space_dir}/{repo_id}/resolve/main/pytorch_model.bin"
labels_txt_url = f"{hf_space_dir}/{repo_id}/resolve/main/class_names.txt"

In [ ]:
response = requests.get(labels_txt_url)
LABELS = []
assert response.status_code == 200
for line in response.iter_lines(decode_unicode=True):
    if line: LABELS.append(line)
state_dict = load_state_dict_from_url(model_url, map_location='cpu')

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding="same"),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding="same"),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding="same"),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(1152, 256),
    nn.ReLU(),
    nn.Linear(256, len(LABELS)),
)
model.load_state_dict(state_dict, strict=False);
model.eval();

### Create the interface

In [ ]:
def predict(input):
    im = input['composite']
    x = torch.tensor(im, dtype=torch.float32)\
        .unsqueeze(0)\
        .unsqueeze(0) / 255.0
    resized_x = F.interpolate(x, size=(28, 28))
    with torch.no_grad():
        out = model(resized_x)
    probabilities = torch.nn.functional.softmax(out[0], dim=0)
    values, indices = torch.topk(probabilities, 5)
    return {LABELS[i]: v.item() for i, v in zip(indices, values)}

In [ ]:
# sample_input = {
#     'composite': np.zeros((28, 28))
# }
# output = predict(sample_input)
# print(output)

In [ ]:
interface = gr.Interface(
    predict,
    inputs=gr.Sketchpad(image_mode="L", height=400, width=400),
    outputs="label",
    title="Sketch Recognition",
    description="Who wants to play Pictionary? Draw a common object like a shovel or a laptop, and the algorithm will guess in real time!",
    article="<p style='text-align: center'>Sketch Recognition | Demo Model</p>",
    live=True,
    # this means the sketch demo makes a prediction every time someone draws on the pad
    # (no submit button!).
)

### Create the link

In [ ]:
# interface.launch(share=True, theme="huggingface")
interface.launch(share=False, theme="huggingface")

# Serve on Hugging Face Spaces

## Creting the space repo files

In [ ]:
current_path = Path().cwd()

local_model_path = hf_hub_download(
    repo_id=repo_id,
    filename="pytorch_model.bin",
    local_dir=current_path / "artifacts",
    repo_type="space"
)
print(f"model state dict saved at {Path(local_model_path).relative_to(current_path)}")


local_labels_path = hf_hub_download(
    repo_id=repo_id,
    filename="class_names.txt",
    local_dir=current_path / "artifacts",
    repo_type="space"
)
print(f"labels text file saved at {Path(local_labels_path).relative_to(current_path)}")

## Write `app.py`

- create a space with gradio backend
- once created, git clone and add `app.py` and `requirements.txt` (TODO: how does gradio detect it?) and all files in `artifacts/` dir
- TODO: find a way to automatically create `requirements.txt` --> torch, gradio with proper versions
- first, try `app.py` locally
- then push to hf space and test online

In [ ]:
%%writefile app.py

from pathlib import Path
import torch
import gradio as gr
from torch import nn
import torch.nn.functional as F

local_model_path = "artifacts/pytorch_model.bin"
local_labels_path = "artifacts/class_names.txt"
LABELS = Path(local_labels_path).read_text().splitlines()

model = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding="same"),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding="same"),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding="same"),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(1152, 256),
    nn.ReLU(),
    nn.Linear(256, len(LABELS)),
)
state_dict = torch.load(local_model_path, map_location="cpu")
model.load_state_dict(state_dict, strict=False)
model.eval()


def predict(input):
    im = input['composite']
    x = torch.tensor(im, dtype=torch.float32)\
        .unsqueeze(0)\
        .unsqueeze(0) / 255.0
    resized_x = F.interpolate(x, size=(28, 28))
    with torch.no_grad():
        out = model(resized_x)
    probabilities = torch.nn.functional.softmax(out[0], dim=0)
    values, indices = torch.topk(probabilities, 5)
    return {LABELS[i]: v.item() for i, v in zip(indices, values)}


interface = gr.Interface(
    predict,
    inputs=gr.Sketchpad(image_mode="L", height=400, width=400),
    outputs="label",
    title="Sketch Recognition",
    description="Who wants to play Pictionary? Draw a common object like a shovel or a laptop, and the algorithm will guess in real time!",
    article="<p style='text-align: center'>Sketch Recognition | Demo Model</p>",
    live=True
)

interface.launch()

## Add dependencies

In [ ]:
! echo -e "torch>=2.9.1\ngradio>=6.14.0" > requirements.txt

The final result is something like [this app](https://huggingface.co/spaces/amin-oj/Minimal-Sketch-Detector)